# 2.3 Silver

Transforms bronze and landing data into clean, typed, analytics-ready tables.

**What this notebook covers:**
- Type casting and derived columns (`age` from `date_of_birth`)
- `PIVOT` to reshape long-format address rows into wide format
- Repairing malformed JSON with `REGEXP_REPLACE` + string surgery
- `FROM_JSON` to parse a JSON string into a typed struct
- `LATERAL VIEW EXPLODE` to normalize a nested array into one row per element

**Inputs:**

| Source | Layer |
|--------|-------|
| `de_lab.bronze.customers` | Bronze |
| `de_lab.landing.v_addresses` | Landing |
| `de_lab.landing.v_orders` | Landing |

**Outputs:**

| Table | Description |
|-------|-------------|
| `de_lab.silver.customers` | Type-cast customer records with derived `age` |
| `de_lab.silver.addresses` | Wide-format addresses (Billing + Shipping per customer) |
| `de_lab.silver.fact_line_items` | One row per order line item |

## Silver Customers

In [0]:
CREATE OR REPLACE TABLE de_lab.silver.customers AS
SELECT
  customer_id,
  customer_name,
  email,
  CAST(date_of_birth AS DATE)                                                          AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(date_of_birth AS DATE)) / 365.25) AS INT)  AS age,
  phone,
  CAST(created_timestamp AS TIMESTAMP)                                                 AS created_timestamp
FROM de_lab.bronze.customers;

In [0]:
SELECT * FROM de_lab.silver.customers LIMIT 10;

## Silver Addresses

Source rows are long-format (one row per `address_type`). `PIVOT` reshapes them into a single wide row per customer with separate columns for `Billing` and `Shipping`.

In [0]:
SELECT *
FROM de_lab.landing.v_addresses
ORDER BY customer_id
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE de_lab.silver.addresses AS
SELECT *
FROM (
  SELECT customer_id, address_type, address_line_1, city, state, postcode
  FROM de_lab.landing.v_addresses
)
PIVOT (
  MAX(address_line_1) AS address_line_1,
  MAX(city)           AS city,
  MAX(state)          AS state,
  MAX(postcode)       AS postcode
  FOR address_type IN ('Billing', 'Shipping')
);

In [0]:
SELECT * FROM de_lab.silver.addresses ORDER BY customer_id LIMIT 10;

## Silver Orders

The raw order files contain malformed JSON — date values are unquoted (`"order_date": 2024-01-01` instead of `"order_date": "2024-01-01"`). We fix this in three steps:

1. **Repair** the JSON string with `REGEXP_REPLACE` + string surgery
2. **Parse** it into a typed struct with `FROM_JSON`
3. **Explode** the nested `items` array into one row per line item with `LATERAL VIEW EXPLODE`

### Step 1 — Repair malformed JSON

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_orders_fixed AS
SELECT
  CASE
    WHEN value RLIKE '"order_date": [0-9]{4}-[0-9]{2}-[0-9]{2}'
    THEN CONCAT(
      SPLIT(value, '"order_date": ')[0],
      '"order_date": "',
      REGEXP_EXTRACT(value, '"order_date": ([0-9]{4}-[0-9]{2}-[0-9]{2})', 1),
      '"',
      REGEXP_REPLACE(
        SPLIT(value, '"order_date": ')[1],
        '^[0-9]{4}-[0-9]{2}-[0-9]{2}', ''
      )
    )
    ELSE value
  END AS fixed_value
FROM de_lab.landing.v_orders;

In [0]:
SELECT * FROM tv_orders_fixed LIMIT 5;

### Step 2 — Infer schema

In [0]:
SELECT SCHEMA_OF_JSON(fixed_value) AS json_schema
FROM tv_orders_fixed
LIMIT 1;

### Step 3 — Parse into typed struct

In [0]:
CREATE OR REPLACE TABLE de_lab.silver.orders_json AS
SELECT
  FROM_JSON(
    fixed_value,
    'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, item_id: STRING, name: STRING, price: DOUBLE, quantity: BIGINT>>, order_date: DATE, order_id: BIGINT, order_status: STRING, total_amount: DOUBLE, transaction_timestamp: TIMESTAMP>'
  ) AS json_value
FROM tv_orders_fixed;

In [0]:
DESCRIBE de_lab.silver.orders_json;

### Step 4 — Explore nested items

Each order can contain multiple line items stored in an array. Facts should be at the lowest grain — one row per item.

In [0]:
SELECT
  json_value.order_id,
  SIZE(json_value.items) AS num_items,
  json_value.items
FROM de_lab.silver.orders_json
WHERE SIZE(json_value.items) > 1
LIMIT 5;

### Step 5 — Explode into fact table

In [0]:
CREATE OR REPLACE TABLE de_lab.silver.fact_line_items AS
SELECT
  json_value.order_id,
  json_value.order_status,
  json_value.customer_id,
  json_value.transaction_timestamp,
  json_value.total_amount,
  json_value.order_date,
  item.item_id,
  item.name     AS item_name,
  item.price,
  item.quantity,
  item.category
FROM de_lab.silver.orders_json
LATERAL VIEW EXPLODE(json_value.items) AS item
WHERE json_value.order_id IS NOT NULL
  AND json_value.customer_id IS NOT NULL;

In [0]:
SELECT * FROM de_lab.silver.fact_line_items LIMIT 10;